## Configuration

Set the mode to analyze and configure filtering options.

In [ ]:
# Configuration: Set Analysis Mode, Judge Model, and Filters
# Set the analysis mode: "baseline" or "taskConfined" or "firewalls_data_abstraction_language_converter" or other custom modes based on folder names in logs/
MODE = "firewalls_data_abstraction_language_converter"  # e.g. Change this to "baseline" to analyze baseline results

# Set the judge model to filter by (REQUIRED)
# Options: "gpt-5", "gpt-5-chat-latest" (when using azure, that is the case for all firewall folders), "claude-sonnet-4", "claude-sonnet-4-5", etc.
JUDGE_MODEL = "gpt-5-chat-latest"  # REQUIRED: Specify the judge model that was used for evaluation

# Set the assistant models to analyze
# Options: None (all models), "gemini" (Gemini models only), "claude" (Claude models only), 
#          "gpt" (GPT models only), or a list like ['gemini_2.5_pro', 'claude_sonnet_4_0']
MODEL_FILTER = None  # Change to None for all models, or specify models to include

# Set the personas to analyze
# Options: None (all personas), or a list like [1, 4] or ['persona1', 'persona4']
PERSONA_FILTER = None  # ALL personas

# Set the use cases to analyze
# Options: None (all use cases), a string like 'travel_planning', or a list
USE_CASE_FILTER = None  # ALL use cases

if not JUDGE_MODEL:
    raise ValueError("JUDGE_MODEL is required! Please specify a judge model (e.g., 'gpt-5', 'claude-sonnet-4')")

print(f"📊 Analysis Configuration:")
print(f"   Mode: {MODE}")
print(f"   Judge Model: {JUDGE_MODEL}")
print(f"   Analysis Path: logs/*/{MODE}/")
if MODEL_FILTER:
    if isinstance(MODEL_FILTER, str):
        print(f"   Assistant Models: {MODEL_FILTER} models only")
    else:
        print(f"   Assistant Models: {MODEL_FILTER}")
else:
    print(f"   Assistant Models: All existing models")
if PERSONA_FILTER:
    print(f"   Personas: {PERSONA_FILTER}")
else:
    print(f"   Personas: All personas")
if USE_CASE_FILTER:
    print(f"   Use Cases: {USE_CASE_FILTER}")
else:
    print(f"   Use Cases: All use cases")

📊 Analysis Configuration:
   Mode: firewalls_data_abstraction_language_converter
   Judge Model: gpt-4o-mini
   Analysis Path: logs/*/firewalls_data_abstraction_language_converter/
   Assistant Models: ['gpt_4o_mini']
   Personas: All personas
   Use Cases: travel_planning


## 1. Import Libraries and Load Data

Import required libraries and load result files from the logs directory.

In [2]:
# Import standard libraries
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Force reload of data_loading module to pick up changes
import importlib
import data_loading
importlib.reload(data_loading)
from data_loading import load_all_results

# Import custom utility modules from results_analysis package
from data_enhancement import create_unified_dataset, enhance_dataset_with_groupings
from analysis_utils import (
    analyze_by_attack_type_and_meta_level,
    generate_per_model_use_case_analysis,
    create_attack_type_comparison
)
from latex_generation import generate_all_latex_tables, write_latex_file

# Constants for attack success evaluation
SECURITY_SUCCESS_VALUES = ['Successful', 'Partially successful']
PRIVACY_SUCCESS_FIELDS = ['shared_raw_data', 'leaked_information']

# Selected model display names for comparison
SELECTED_MODEL_NAMES = {
    'gpt-4o': 'GPT-4o',
    'claude_sonnet_4_0': 'Claude Sonnet 4*',
    'grok_3': 'Grok 3',
    'o3_mini': 'o3-mini*',
    'gemini_2.5_flash': 'Gemini 2.5 Flash*',
    'gemini_2.5_pro': 'Gemini 2.5 Pro',
    'claude_3_5_haiku_latest': 'Claude 3.5 Haiku',
    'gpt_5_chat': 'GPT-5*'
}

print("✅ Libraries imported and modules loaded")

# Load all results filtered by MODE, JUDGE_MODEL, MODEL_FILTER, PERSONA_FILTER, and USE_CASE_FILTER
# Loads result files from logs directory (one level up from results_analysis folder)
df = load_all_results(
    logs_dir='../logs', 
    mode=MODE, 
    judge_model=JUDGE_MODEL, 
    model_filter=MODEL_FILTER,
    persona_filter=PERSONA_FILTER,
    use_case_filter=USE_CASE_FILTER,
    verbose=False
)

# Display loading summary
filter_desc = f", judge model: {JUDGE_MODEL}"
if MODEL_FILTER:
    if isinstance(MODEL_FILTER, str):
        filter_desc += f", model filter: {MODEL_FILTER}"
    else:
        filter_desc += f", models: {MODEL_FILTER}"
if PERSONA_FILTER:
    filter_desc += f", personas: {PERSONA_FILTER}"
if USE_CASE_FILTER:
    filter_desc += f", use_case: {USE_CASE_FILTER}"

print(f"\n✅ Loaded {len(df)} results for mode: {MODE}{filter_desc}")
if len(df) > 0:
    print(f"   Use cases: {sorted(df['use_case'].unique())}")
    print(f"   Models: {sorted(df['model'].unique())}")
    print(f"   Personas: {sorted(df['persona'].unique())}")
else:
    print(f"   ⚠️  No results found for mode '{MODE}' and judge model '{JUDGE_MODEL}'")

✅ Libraries imported and modules loaded

✅ Loaded 2 results for mode: firewalls_data_abstraction_language_converter, judge model: gpt-4o-mini, models: ['gpt_4o_mini'], use_case: travel_planning
   Use cases: ['travel_planning']
   Models: ['gpt_4o_mini']
   Personas: ['persona1']


## 2. Create Enhanced Unified Dataset

Create a unified dataset combining utility, security, and privacy evaluations with attack details from resources.

In [3]:
# Create the enhanced unified dataset
# This combines utility judge data with security/privacy judge data
# and enriches with attack details from resource files
print("🔄 Creating enhanced unified dataset...")
enhanced_df = create_unified_dataset(df, verbose=True)

# Add attack groupings based on attack characteristics
enhanced_df = enhance_dataset_with_groupings(enhanced_df)

print(f"✅ Dataset created: {len(enhanced_df)} scenarios across {len(enhanced_df['model'].unique())} models and {len(enhanced_df['use_case'].unique())} use cases")

🔄 Creating enhanced unified dataset...
🔍 load_all_attack_files: Loaded 24 files, 0 errors
   Use cases: ['insurance', 'real_estate', 'travel_planning']
✅ Dataset created: 1 scenarios across 1 models and 1 use cases


## 3. Statistical Analysis

Generate detailed CSV exports with confidence intervals across all meta-levels.

In [4]:
# Create output directory
output_base_dir = Path('analysis_outputs')
output_base_dir.mkdir(exist_ok=True)

print("📊 === CREATING CSV EXPORTS === 📊")
print("="*80)

# Define meta-levels to analyze
global_meta_levels = ['model', 'attack_type']
model_filtered_meta_levels = ['use_case', 'responsibility_flag', 'privacy_data_category', 'attack_name_group']
all_meta_levels = global_meta_levels + model_filtered_meta_levels

# Analyze each attack type separately
# Creates comprehensive CSV files with confidence intervals for all meta-levels
all_results = {}
for attack_type in ['security', 'privacy', 'benign']:
    results = analyze_by_attack_type_and_meta_level(
        enhanced_df, attack_type, all_meta_levels, output_base_dir, verbose=True
    )
    all_results[attack_type] = results

# Create attack type comparison
create_attack_type_comparison(enhanced_df, output_base_dir, verbose=True)

# Generate per-model use_case aggregation CSVs for LaTeX table generation
generate_per_model_use_case_analysis(enhanced_df, output_base_dir, verbose=True)

print(f"\n🎉 Analysis Complete!")
print(f"📁 All CSV outputs saved in 'analysis_outputs' directory")

📊 === CREATING CSV EXPORTS === 📊

🎯 === SECURITY ATTACK ANALYSIS === 🎯
Total security scenarios: 1
------------------------------------------------------------

📊 Model Analysis for Security Attacks (Global):

📊 Attack Type Analysis for Security Attacks (Global):
⚠️  No privacy data found
⚠️  No benign data found

📊 Creating attack type comparison CSV...
  💾 Saved: attack_type_comparison.csv

📊 Generating per-model use_case analysis CSVs...
  💾 Saved: security_use_case_gpt_4o_mini_analysis.csv

🎉 Analysis Complete!
📁 All CSV outputs saved in 'analysis_outputs' directory


## 4. Generate LaTeX Tables

Generate publication-ready LaTeX tables from the analysis results.

In [5]:
print("🚀 Generating Selected LaTeX Tables for Paper...")
print("="*80)

# Determine model groups from loaded data
available_models = sorted(df['model'].unique())
PRIVACY_ALL_MODELS = available_models

# For complete models, use those that appear in all three use cases
if len(df) > 0:
    model_use_case_counts = df.groupby('model')['use_case'].nunique()
    complete_models = model_use_case_counts[model_use_case_counts >= 3].index.tolist()
    PRIVACY_COMPLETE_MODELS = sorted(complete_models) if complete_models else available_models
else:
    PRIVACY_COMPLETE_MODELS = available_models

SECURITY_ALL_MODELS = PRIVACY_ALL_MODELS
SECURITY_COMPLETE_MODELS = PRIVACY_COMPLETE_MODELS

print(f"📋 Model groups configured:")
print(f"   All models: {PRIVACY_ALL_MODELS}")
print(f"   Complete models: {PRIVACY_COMPLETE_MODELS}")

# Generate all LaTeX tables
latex_selected = generate_all_latex_tables(
    df, output_base_dir, MODE, JUDGE_MODEL, MODEL_FILTER,
    PRIVACY_ALL_MODELS, PRIVACY_COMPLETE_MODELS,
    SECURITY_ALL_MODELS, SECURITY_COMPLETE_MODELS,
    SELECTED_MODEL_NAMES
)

# Write to file in root directory (one level up from results_analysis)
output_file = Path('..') / 'latex_tables_selected.tex'
write_latex_file(latex_selected, output_file, MODE, JUDGE_MODEL, MODEL_FILTER)

print(f"\n📝 LaTeX tables written to {output_file}")
print("\n🎉 All selected LaTeX tables generated successfully!")

🚀 Generating Selected LaTeX Tables for Paper...
📋 Model groups configured:
   All models: ['gpt_4o_mini']
   Complete models: ['gpt_4o_mini']

📝 LaTeX tables written to ..\latex_tables_selected.tex

🎉 All selected LaTeX tables generated successfully!


## 5. Cleanup (Optional)

Remove temporary analysis files to keep the workspace clean. 
**Comment out this cell if you want to keep and inspect intermediate results.**

In [6]:
# Clean up analysis_outputs directory
import shutil
if output_base_dir.exists():
    print(f"🧹 Cleaning up temporary directory: {output_base_dir}")
    try:
        shutil.rmtree(output_base_dir)
        print(f"   ✅ Successfully removed {output_base_dir}")
    except Exception as e:
        print(f"   ⚠️  Error removing directory: {e}")
else:
    print(f"ℹ️  Directory {output_base_dir} does not exist (already cleaned up)")

print("\n🎉 Analysis complete and workspace cleaned!")

🧹 Cleaning up temporary directory: analysis_outputs
   ✅ Successfully removed analysis_outputs

🎉 Analysis complete and workspace cleaned!
